In [ ]:
# CC DATA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [ ]:
cc_path = r"C:\Users\Jayas\Downloads\CC_Modeling.csv"

cc = pd.read_csv(cc_path, low_memory=False)

In [ ]:
cc.columns = cc.columns.str.strip()

In [ ]:
cc_df = cc[[
    "final_disease_category",
    "prscrbr_geo_desc",
    "data_year",
    "tot_drug_cst"
]].copy()

cc_df = cc_df.rename(columns={
    "final_disease_category": "Disease Category",
    "prscrbr_geo_desc": "State",
    "data_year": "Year",
    "tot_drug_cst": "Cost"
})

In [ ]:
cc_df["Year"] = pd.to_numeric(cc_df["Year"], errors="coerce")
cc_df["Cost"] = pd.to_numeric(cc_df["Cost"], errors="coerce")

cc_df = cc_df.dropna(subset=["Disease Category", "State", "Year", "Cost"])

In [ ]:
cc_df = cc_df[cc_df["Year"].between(2017, 2023)]

In [ ]:
cc_grouped = (
    cc_df.groupby(["Disease Category", "State", "Year"], as_index=False)["Cost"]
    .sum()
)

In [ ]:
def fit_cc_trend(group):

    group = group.sort_values("Year")

    # IMPORTANT FIX
    if len(group) < 2:
        return pd.Series({
            "Cost Trend Slope": np.nan,
            "Intercept": np.nan,
            "R_Squared": np.nan,
            "Projected_2024": np.nan,
            "Projected_2025": np.nan,
            "Projected_2026": np.nan,
            "Projected_2027": np.nan,
            "WhatIf_Adjusted_2027": np.nan
        })

    X = group[["Year"]].values
    y = group["Cost"].values

    model = LinearRegression()
    model.fit(X, y)

    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)

    future_years = np.array([2024, 2025, 2026, 2027]).reshape(-1, 1)
    future_preds = model.predict(future_years)

    whatif_2027 = future_preds[3] * 1.05

    return pd.Series({
        "Cost Trend Slope": model.coef_[0],
        "Intercept": model.intercept_,
        "R_Squared": r2,
        "Projected_2024": future_preds[0],
        "Projected_2025": future_preds[1],
        "Projected_2026": future_preds[2],
        "Projected_2027": future_preds[3],
        "WhatIf_Adjusted_2027": whatif_2027
    })

In [ ]:
cc_results = (
    cc_grouped.groupby(["Disease Category", "State"], group_keys=False)
    .apply(fit_cc_trend)
    .reset_index()
)

cc_results = cc_results[cc_results["R_Squared"] >= 0.70].copy()

cc_results = cc_results.sort_values("Cost Trend Slope", ascending=False)

cols_to_format = [
    "Cost Trend Slope",
    "Projected_2024",
    "Projected_2025",
    "Projected_2026",
    "Projected_2027",
    "WhatIf_Adjusted_2027"
]

cc_results[cols_to_format] = cc_results[cols_to_format] / 1_000_000
cc_results[cols_to_format] = cc_results[cols_to_format].round(2)

cc_results["R_Squared"] = cc_results["R_Squared"].round(2)

C:\Users\Jayas\AppData\Local\Temp\ipykernel_15684\1285896413.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(fit_cc_trend)


In [ ]:
cc_national = cc_results[cc_results["State"] == "National"]
cc_states = cc_results[cc_results["State"] != "National"]

cc_national_top20 = cc_national.head(20)
cc_states_top20 = cc_states.head(20)

In [ ]:
print("===== NATIONAL TOP 20 =====")
display(cc_national_top20)

print("===== STATES TOP 20 =====")
display(cc_states_top20)

===== NATIONAL TOP 20 =====


,Disease Category,State,Cost Trend Slope,Intercept,R_Squared,Projected_2024,Projected_2025,Projected_2026,Projected_2027,WhatIf_Adjusted_2027
405,Gastroenterology/Endocrinology,National,5964.63,-1.200318e+13,0.93,69223.21,75187.84,81152.46,87117.09,91472.95
912,Oncology/Immunology,National,5717.73,-1.150835e+13,1.00,64335.45,70053.18,75770.91,81488.64,85563.07
518,Hematology/Cardiovascular,National,3137.35,-6.321524e+12,1.00,28474.98,31612.33,34749.68,37887.03,39781.38
1233,Respiratory,National,1035.65,-2.079130e+12,0.90,17019.11,18054.76,19090.41,20126.06,21132.36
165,Dermatology,National,882.80,-1.774616e+12,0.87,12167.00,13049.80,13932.59,14815.39,15556.16
1145,Psychiatry/Neurology,National,829.23,-1.651273e+12,0.82,27080.29,27909.52,28738.74,29567.97,31046.37
104,Cardiovascular,National,682.43,-1.360541e+12,0.72,20701.54,21383.98,22066.41,22748.84,23886.28
573,Immunology,National,322.50,-6.490321e+11,1.00,3718.03,4040.53,4363.04,4685.54,4919.82
973,Ophthalmology/ENT,National,221.95,-4.458493e+11,0.98,3376.26,3598.21,3820.16,4042.11,4244.21
346,Gastroenterology,National,219.77,-4.422671e+11,0.97,2549.43,2769.20,2988.98,3208.75,3369.18


===== STATES TOP 20 =====


,Disease Category,State,Cost Trend Slope,Intercept,R_Squared,Projected_2024,Projected_2025,Projected_2026,Projected_2027,WhatIf_Adjusted_2027
887,Oncology/Immunology,California,545.48,-1.097582e+12,1.00,6469.97,7015.45,7560.93,8106.41,8511.74
380,Gastroenterology/Endocrinology,California,527.03,-1.060549e+12,0.93,6161.67,6688.70,7215.73,7742.76,8129.90
424,Gastroenterology/Endocrinology,Texas,499.14,-1.004718e+12,0.94,5545.70,6044.84,6543.98,7043.12,7395.28
411,Gastroenterology/Endocrinology,New York,448.65,-9.027054e+11,0.95,5352.22,5800.86,6249.51,6698.15,7033.06
918,Oncology/Immunology,New York,446.53,-8.987785e+11,0.99,5006.83,5453.37,5899.90,6346.43,6663.76
385,Gastroenterology/Endocrinology,Florida,406.07,-8.172688e+11,0.92,4611.91,5017.97,5424.04,5830.11,6121.62
892,Oncology/Immunology,Florida,380.33,-7.653646e+11,1.00,4424.48,4804.81,5185.14,5565.47,5843.75
931,Oncology/Immunology,Texas,371.04,-7.467991e+11,1.00,4186.36,4557.40,4928.44,5299.48,5564.45
418,Gastroenterology/Endocrinology,Pennsylvania,257.71,-5.183543e+11,0.96,3248.42,3506.13,3763.84,4021.55,4222.62
498,Hematology/Cardiovascular,Florida,250.90,-5.055702e+11,1.00,2245.60,2496.50,2747.40,2998.29,3148.21


In [ ]:
cc_national_top20.to_excel(
    r"C:\Users\Jayas\Downloads\CC_National_Top20.xlsx",
    index=False
)

cc_states_top20.to_excel(
    r"C:\Users\Jayas\Downloads\CC_State_Top20.xlsx",
    index=False
)

print("Both Excel files saved successfully")

Both Excel files saved successfully


In [ ]:
cc_path = r"C:\Users\Jayas\Downloads\CC_Modeling.csv"
imp_path = r"C:\Users\Jayas\Downloads\IMP_Modeling.csv"

cc = pd.read_csv(cc_path)
imp = pd.read_csv(imp_path)

print("CC shape:", cc.shape)
print("IMP shape:", imp.shape)

C:\Users\Jayas\AppData\Local\Temp\ipykernel_15684\430983897.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  imp = pd.read_csv(imp_path)


CC shape: (465769, 45)
IMP shape: (800556, 52)


In [ ]:
# IMP DATA

In [ ]:
imp_df = imp.copy()

imp_df = imp_df.rename(columns={
    "data_year": "Year",
    "tot_drug_cst": "Cost",
    "tot_clms": "Claims",
    "tot_30day_fills": "Fills"
})

In [ ]:
imp_df["Disease Category"] = imp_df["final_disease_category"]
imp_df["State"] = imp_df["prscrbr_geo_desc"]

In [ ]:
imp_df["Year"] = pd.to_numeric(imp_df["Year"], errors="coerce")
imp_df["Cost"] = pd.to_numeric(imp_df["Cost"], errors="coerce")

imp_df = imp_df.dropna(subset=["Year", "Cost", "Disease Category", "State"])
imp_df = imp_df[imp_df["Year"].between(2017, 2023)]

In [ ]:
imp_grouped = (
    imp_df.groupby(["Disease Category", "State", "Year"], as_index=False)["Cost"]
    .sum()
)

In [ ]:
def fit_imp_trend(group):
    if len(group) < 2:
        return pd.Series({
            "Cost Trend Slope": np.nan,
            "R_Squared": np.nan,
            "Projected_2024": np.nan,
            "Projected_2025": np.nan,
            "Projected_2026": np.nan,
            "Projected_2027": np.nan,
            "WhatIf_2027": np.nan
        })

    X = group["Year"].values.reshape(-1, 1)
    y = group["Cost"].values

    model = LinearRegression()
    model.fit(X, y)

    preds = model.predict(X)
    r2 = r2_score(y, preds)

    future_years = np.array([[2024], [2025], [2026], [2027]])
    future_preds = model.predict(future_years)

    return pd.Series({
        "Cost Trend Slope": model.coef_[0],
        "R_Squared": r2,
        "Projected_2024": future_preds[0],
        "Projected_2025": future_preds[1],
        "Projected_2026": future_preds[2],
        "Projected_2027": future_preds[3],
        "WhatIf_2027": future_preds[3] * 1.05
    })

In [ ]:
imp_results = (
    imp_grouped.groupby(["Disease Category", "State"], group_keys=False)
    .apply(fit_imp_trend)
    .reset_index()
)

imp_results = imp_results[imp_results["R_Squared"] >= 0.70]
imp_results = imp_results.sort_values("Cost Trend Slope", ascending=False)

C:\Users\Jayas\AppData\Local\Temp\ipykernel_15684\1681453084.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(fit_imp_trend)


In [ ]:
cols = [
    "Cost Trend Slope",
    "Projected_2024",
    "Projected_2025",
    "Projected_2026",
    "Projected_2027",
    "WhatIf_2027"
]

imp_results[cols] = imp_results[cols] / 1_000_000
imp_results[cols] = imp_results[cols].round(2)

imp_results["R_Squared"] = imp_results["R_Squared"].round(2)

In [ ]:
imp_national = imp_results[imp_results["State"] == "National"]
imp_states = imp_results[imp_results["State"] != "National"]

imp_national_top20 = imp_national.head(20)
imp_states_top20 = imp_states.head(20)

print("===== IMP NATIONAL TOP 20 =====")
display(imp_national_top20)

print("===== IMP STATE TOP 20 =====")
display(imp_states_top20)

===== IMP NATIONAL TOP 20 =====


,Disease Category,State,Cost Trend Slope,R_Squared,Projected_2024,Projected_2025,Projected_2026,Projected_2027,WhatIf_2027
448,Gastroenterology/Endocrinology,National,5963.62,0.93,69322.41,75286.03,81249.65,87213.27,91573.94
990,Oncology/Immunology,National,5732.79,1.00,64488.01,70220.80,75953.59,81686.38,85770.70
567,Hematology/Cardiovascular,National,3137.41,1.00,28475.38,31612.80,34750.21,37887.62,39782.00
1347,Respiratory,National,1031.77,0.90,17014.88,18046.65,19078.42,20110.19,21115.70
204,Dermatology,National,883.90,0.87,12174.59,13058.48,13942.38,14826.28,15567.59
1230,Psychiatry/Neurology,National,833.31,0.83,27165.80,27999.11,28832.43,29665.74,31149.02
142,Cardiovascular,National,685.56,0.72,20738.35,21423.91,22109.48,22795.04,23934.79
628,Immunology,National,323.47,1.00,3724.16,4047.63,4371.09,4694.56,4929.29
1052,Ophthalmology/ENT,National,222.12,0.98,3378.07,3600.19,3822.31,4044.43,4246.66
387,Gastroenterology,National,221.15,0.97,2557.94,2779.09,3000.24,3221.39,3382.46


===== IMP STATE TOP 20 =====


,Disease Category,State,Cost Trend Slope,R_Squared,Projected_2024,Projected_2025,Projected_2026,Projected_2027,WhatIf_2027
965,Oncology/Immunology,California,606.95,1.00,6927.52,7534.47,8141.43,8748.38,9185.80
423,Gastroenterology/Endocrinology,California,531.89,0.93,6214.61,6746.50,7278.38,7810.27,8200.78
467,Gastroenterology/Endocrinology,Texas,501.84,0.94,5577.04,6078.88,6580.71,7082.55,7436.68
996,Oncology/Immunology,New York,483.49,0.99,5319.32,5802.81,6286.29,6769.78,7108.27
454,Gastroenterology/Endocrinology,New York,451.09,0.95,5393.38,5844.47,6295.57,6746.66,7083.99
970,Oncology/Immunology,Florida,410.07,1.00,4676.22,5086.29,5496.36,5906.43,6201.76
428,Gastroenterology/Endocrinology,Florida,408.88,0.93,4643.27,5052.16,5461.04,5869.93,6163.42
1009,Oncology/Immunology,Texas,394.53,1.00,4396.64,4791.17,5185.71,5580.24,5859.25
1003,Oncology/Immunology,Pennsylvania,271.61,1.00,3085.50,3357.11,3628.72,3900.34,4095.35
461,Gastroenterology/Endocrinology,Pennsylvania,259.78,0.96,3279.86,3539.64,3799.41,4059.19,4262.15


In [ ]:
# COMPARISON

In [ ]:
cc_compare = cc_results[[
    "Disease Category",
    "State",
    "Cost Trend Slope"
]]

imp_compare = imp_results[[
    "Disease Category",
    "State",
    "Cost Trend Slope"
]]

In [ ]:
comparison = pd.merge(
    cc_compare,
    imp_compare,
    on=["Disease Category", "State"],
    how="inner",
    suffixes=("_CC", "_IMP")
)

In [ ]:
comparison["Trend_Difference"] = (
    comparison["Cost Trend Slope_CC"] -
    comparison["Cost Trend Slope_IMP"]
)

comparison = comparison.sort_values("Trend_Difference", ascending=False)

In [ ]:
comparison_national = comparison[comparison["State"] == "National"]
comparison_states = comparison[comparison["State"] != "National"]

In [ ]:
cols = ["Cost Trend Slope_CC", "Cost Trend Slope_IMP", "Trend_Difference"]

comparison[cols] = comparison[cols] / 1_000_000
comparison[cols] = comparison[cols].round(2)

In [ ]:
comparison_national_top20 = comparison_national.head(20)
comparison_states_top20 = comparison_states.head(20)

In [ ]:
print("===== CC vs IMP (NATIONAL) =====")
display(comparison_national_top20)

print("===== CC vs IMP (STATE LEVEL) =====")
display(comparison_states_top20)

===== CC vs IMP (NATIONAL) =====


,Disease Category,State,Cost Trend Slope_CC,Cost Trend Slope_IMP,Trend_Difference
3,Respiratory,National,0.00,0.00,0.0
0,Gastroenterology/Endocrinology,National,0.01,0.01,0.0
655,Nephrology,National,-0.00,-0.00,0.0
2,Hematology/Cardiovascular,National,0.00,0.00,-0.0
653,Analgesics,National,-0.00,-0.00,-0.0
654,Fluids/Electrolytes,National,-0.00,-0.00,-0.0
20,Ophthalmology/ENT,National,0.00,0.00,-0.0
15,Immunology,National,0.00,0.00,-0.0
4,Dermatology,National,0.00,0.00,-0.0
22,Gastroenterology,National,0.00,0.00,-0.0


===== CC vs IMP (STATE LEVEL) =====


,Disease Category,State,Cost Trend Slope_CC,Cost Trend Slope_IMP,Trend_Difference
249,Dermatology,Arkansas,0.0,0.0,0.0
179,Dermatology,Indiana,0.0,0.0,0.0
67,Respiratory,Florida,0.0,0.0,0.0
89,Respiratory,New York,0.0,0.0,0.0
189,Respiratory,Colorado,0.0,0.0,0.0
100,Respiratory,Pennsylvania,0.0,0.0,0.0
227,Dermatology,Louisiana,0.0,0.0,0.0
106,Respiratory,California,0.0,0.0,0.0
121,Respiratory,North Carolina,0.0,0.0,0.0
375,Hematology,Minnesota,0.0,0.0,0.0
